In [1]:
from scripts.Utils import NER_Utils
import torch
from transformers import RobertaForTokenClassification, RobertaTokenizerFast, TrainingArguments, Trainer
from scripts.Reader import obtain_dataset, obtain_label_list

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    device = torch.device('cuda')
print("Current Device:", torch.cuda.current_device(), torch.cuda.get_device_name(torch.cuda.current_device()))

CUDA available: True
Current Device: 0 NVIDIA GeForce RTX 4070 Ti


In [2]:
datasets, label_list, label2id, id2label = obtain_dataset("TempEval3", "BIO")

In [ ]:
# Load tokenizer and model
model_name = 'roberta-base'
tokenizer = RobertaTokenizerFast.from_pretrained(model_name, add_prefix_space=True, use_fast=True)
model = RobertaForTokenClassification.from_pretrained(model_name, num_labels=len(label_list), label2id=label2id, id2label=id2label)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

d:\GeoTKG\GeoTKG\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Harry\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
training_args = TrainingArguments(
    output_dir="./results/EventTimex-NER",
    logging_dir="./logs/EventTimex-NER",
    eval_strategy="steps",
    save_strategy="steps",
    logging_steps=100,
    num_train_epochs=1,
    save_total_limit=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

In [5]:
utils = NER_Utils(tokenizer, label_list)
datasets = utils.tokenize_datasets(datasets)

Map:   0%|          | 0/22865 [00:00<?, ? examples/s]

Map:   0%|          | 0/3326 [00:00<?, ? examples/s]

Map:   0%|          | 0/223 [00:00<?, ? examples/s]

In [6]:
datasets

DatasetDict({
    train: Dataset({
        features: ['tokens', 'label', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 22865
    })
    eval: Dataset({
        features: ['tokens', 'label', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3326
    })
    test: Dataset({
        features: ['tokens', 'label', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 223
    })
})

In [7]:
timex3_ner = Trainer(
    model=model,
    args=training_args,
    compute_metrics=utils.compute_metrics,
    data_collator=utils.data_collator,
    tokenizer=tokenizer,
    train_dataset=datasets["train"],
    eval_dataset=datasets["eval"],
)

C:\Users\Harry\AppData\Local\Temp\ipykernel_25744\716834860.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  timex3_ner = Trainer(


In [8]:
timex3_ner.train()

Step,Training Loss,Validation Loss,Precision,Recall,F1
100,0.309100,0.129184,0.861411,0.843719,0.852473
200,0.100400,0.128560,0.864262,0.877075,0.870622
300,0.084700,0.124146,0.873519,0.865493,0.869488
400,0.085600,0.127247,0.857176,0.859161,0.858168
500,0.079500,0.122046,0.896198,0.829975,0.861816
600,0.076500,0.118226,0.865113,0.888426,0.876614
700,0.076200,0.111221,0.885971,0.867501,0.876639
800,0.071500,0.110787,0.876705,0.878465,0.877584
900,0.070800,0.118586,0.863083,0.892672,0.877628
1000,0.065500,0.119392,0.890072,0.858389,0.873944


TrainOutput(global_step=1430, training_loss=0.09055817485689283, metrics={'train_runtime': 483.187, 'train_samples_per_second': 47.321, 'train_steps_per_second': 2.96, 'total_flos': 5100424470957516.0, 'train_loss': 0.09055817485689283, 'epoch': 1.0})

In [9]:
timex3_ner.evaluate(datasets["test"])

{'eval_loss': 0.15938691794872284,
 'eval_precision': 0.8491879350348028,
 'eval_recall': 0.8280542986425339,
 'eval_f1': 0.838487972508591,
 'eval_runtime': 0.5121,
 'eval_samples_per_second': 435.445,
 'eval_steps_per_second': 13.669,
 'epoch': 1.0}

In [10]:
timex3_ner.save_model("./results/EventTimex-NER/final_model")
tokenizer.save_pretrained("./results/EventTimex-NER/final_model")


('./results/EventTimex-NER/final_model\\tokenizer_config.json',
 './results/EventTimex-NER/final_model\\special_tokens_map.json',
 './results/EventTimex-NER/final_model\\vocab.json',
 './results/EventTimex-NER/final_model\\merges.txt',
 './results/EventTimex-NER/final_model\\added_tokens.json',
 './results/EventTimex-NER/final_model\\tokenizer.json')